# Cost Analysis Notebook

This notebook loads the Chinook sample SQLite database for cost/sales analysis.

The `chinook.db` file lives alongside this notebook in the project root.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path("chinook.db")
assert DB_PATH.exists(), f"Database not found at {DB_PATH.resolve()}"

conn = sqlite3.connect(DB_PATH)
conn

## Inspect the schema

List every table in the database with its row count.

In [2]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    conn,
)

table_summary = pd.DataFrame(
    {
        "table": tables["name"],
        "row_count": [
            pd.read_sql_query(f'SELECT COUNT(*) AS n FROM "{t}"', conn)["n"].iloc[0]
            for t in tables["name"]
        ],
    }
)
table_summary

,table,row_count
0,Album,347
1,Artist,275
2,Customer,59
3,Employee,8
4,Genre,25
5,Invoice,412
6,InvoiceLine,2240
7,MediaType,5
8,Playlist,18
9,PlaylistTrack,8715


## Load core tables into DataFrames

Pull the tables most useful for cost/sales analysis into memory.

In [3]:
invoices = pd.read_sql_query("SELECT * FROM Invoice", conn, parse_dates=["InvoiceDate"])
invoice_items = pd.read_sql_query("SELECT * FROM InvoiceLine", conn)
tracks = pd.read_sql_query("SELECT * FROM Track", conn)
customers = pd.read_sql_query("SELECT * FROM Customer", conn)

print(f"invoices:      {invoices.shape}")
print(f"invoice_items: {invoice_items.shape}")
print(f"tracks:        {tracks.shape}")
print(f"customers:     {customers.shape}")

invoices.head()

invoices:      (412, 9)
invoice_items: (2240, 5)
tracks:        (3503, 9)
customers:     (59, 13)


,InvoiceId,CustomerId,InvoiceDate,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total
0,1,2,2021-01-01,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,1.98
1,2,4,2021-01-02,Ullevålsveien 14,Oslo,None,Norway,0171,3.96
2,3,8,2021-01-03,Grétrystraat 63,Brussels,None,Belgium,1000,5.94
3,4,14,2021-01-06,8210 111 ST NW,Edmonton,AB,Canada,T6G 2C7,8.91
4,5,23,2021-01-11,69 Salem Street,Boston,MA,USA,2113,13.86
